In [45]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

# Step1:Gathering Data

In [46]:
df=pd.read_csv('train.txt' ,sep=';',header=None,names=['text','emotions']) 
df

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [47]:
df.isnull().sum()

text        0
emotions    0
dtype: int64

In [48]:
unique_emotions=df['emotions'].unique()
unique_emotions

<ArrowStringArray>
['sadness', 'anger', 'love', 'surprise', 'fear', 'joy']
Length: 6, dtype: str

In [49]:
unique_emotions = df['emotions'].unique()

emotion_numbers = {}

i = 0

for emo in unique_emotions:
    emotion_numbers[emo] = i
    i += 1

df['emotions'] = df['emotions'].map(emotion_numbers)
df

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


## Converting all test into lower case

In [50]:
df['text']=df['text'].apply(lambda x:x.lower())
df.head()

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


## Removing puncuations

In [51]:
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

df['text']=df['text'].apply(remove_punc)


In [52]:
df.head()


,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


## Remove numbers

In [53]:
def remove_no(text):
    new_text="" 
    for i in text:
        if not i.isdigit():
            new_text=new_text+i 
            
    return new_text

df['text']=df['text'].apply(remove_no)

In [54]:
df.head()

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


## Remove urls

In [55]:
import re
def remove_urls(text):
    plain_txt=re.sub(r'https?://\S+|www\.\S+','',text)
    return plain_txt

df['text']=df['text'].apply(remove_urls)

In [56]:
df.head()

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


## Remove html

In [57]:
from bs4 import BeautifulSoup

def remove_html(text):
    return BeautifulSoup(text, "html.parser").get_text()

ModuleNotFoundError: No module named 'bs4'

## Remove Emojis


In [ ]:
def remove_emojis(text):
    new_text = ""
    
    for i in text:
        if i.isascii():
            new_text = new_text + i
            
    return new_text

## Removing stop words

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HOME\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    new_text = ""
    
    for i in text.split():
        if i.lower() not in stop_words:
            new_text = new_text + i + " "
    
    return new_text.strip()

df['text'] = df['text'].apply(remove_stopwords)

In [ ]:
df

,text,emotions
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
15995,brief time beanbag said anna feel like beaten,0
15996,turning feel pathetic still waiting tables sub...,0
15997,feel strong good overall,5
15998,feel like rude comment im glad,1


## Bag of words

In [58]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['emotions'],
    test_size=0.20,
    random_state=42
)

In [59]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [ ]:
bow_vectorizer=CountVectorizer()

In [71]:
X_train_bow=bow_vectorizer.fit_transform(X_train)
X_test_bow=bow_vectorizer.transform(X_test)


In [76]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_model=MultinomialNB()
nb_model.fit(X_train_bow,y_train)

pred_bow=nb_model.predict(X_test_bow)


print(pred_bow)
print("Accuracy is:",accuracy_score(pred_bow,y_test))


[0 5 0 ... 5 5 0]
Accuracy is: 0.7390625


## TF-IDF

In [80]:
tfidf_vectorizer=TfidfVectorizer()

X_train_tfidf=tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf=tfidf_vectorizer.transform(X_test)


In [82]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_model=MultinomialNB()
nb_model.fit(X_train_tfidf,y_train)

pred_bow=nb_model.predict(X_test_tfidf)


print(pred_bow)
print("Accuracy is:",accuracy_score(pred_bow,y_test))


[0 5 0 ... 5 5 0]
Accuracy is: 0.6175


## Logistic Regression

In [83]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_bow, y_train)

pred_lr = lr_model.predict(X_test_bow)

In [84]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred_lr)

print("Accuracy:", accuracy)

Accuracy: 0.883125
